# Bayesian Network Basics: Construction and Inspection

This notebook covers the fundamentals of working with Bayesian networks in
conin: defining states, building conditional probability distributions (CPDs),
assembling a network, and inspecting its structure.

See also the [constraints](constraints.ipynb) and
[inference](inference.ipynb) notebooks for how to add constraints and run
MAP queries on Bayesian networks.

In [ ]:
from conin.bayesian_network import DiscreteBayesianNetwork, DiscreteCPD

## Defining States

A Bayesian network starts with a set of nodes, each having a finite list of
possible states. States are specified as a dictionary mapping node names to
their state lists.

In [ ]:
bn = DiscreteBayesianNetwork()

bn.states = {
    "Pollution": ["low", "high"],
    "Smoker": ["yes", "no"],
    "Cancer": ["yes", "no"],
    "Xray": ["positive", "negative"],
    "Dyspnoea": ["yes", "no"],
}

print("Nodes:          ", sorted(bn.states.keys()))
print("States of Cancer:", bn.states_of("Cancer"))
print("Card of Xray:   ", bn.card("Xray"))

## Building CPDs

Each node has a conditional probability distribution (CPD) that defines
P(node | parents). A `DiscreteCPD` takes:

- `node` — the node this CPD belongs to
- `parents` — an ordered list of parent nodes (omit for root nodes)
- `values` — the probability table

### Root nodes (no parents)

For root nodes, `values` is a dict mapping each state to its prior
probability, or a flat list in the same order as the state list.

In [ ]:
cpd_pollution = DiscreteCPD(
    node="Pollution",
    values={"low": 0.9, "high": 0.1},
)

cpd_smoker = DiscreteCPD(
    node="Smoker",
    values={"yes": 0.3, "no": 0.7},
)

### Nodes with one parent

For a node with a single parent, `values` is a dict keyed by the parent
state. Each value is itself a dict (or list) of probabilities for the
child node's states.

In [ ]:
cpd_xray = DiscreteCPD(
    node="Xray",
    parents=["Cancer"],
    values={
        "yes": {"positive": 0.9, "negative": 0.1},
        "no":  {"positive": 0.2, "negative": 0.8},
    },
)

cpd_dyspnoea = DiscreteCPD(
    node="Dyspnoea",
    parents=["Cancer"],
    values={
        "yes": {"yes": 0.65, "no": 0.35},
        "no":  {"yes": 0.3,  "no": 0.7},
    },
)

### Nodes with multiple parents

When a node has multiple parents, the `values` dict is keyed by tuples of
parent states (in the same order as the `parents` list).

In [ ]:
cpd_cancer = DiscreteCPD(
    node="Cancer",
    parents=["Smoker", "Pollution"],
    values={
        ("yes", "low"):  {"yes": 0.03,  "no": 0.97},
        ("yes", "high"): {"yes": 0.05,  "no": 0.95},
        ("no",  "low"):  {"yes": 0.001, "no": 0.999},
        ("no",  "high"): {"yes": 0.02,  "no": 0.98},
    },
)

## Assembling the Network

Assign the CPDs to the network via the `cpds` property. The network infers
its edges from the parent relationships in the CPDs — there is no need to
specify edges explicitly.

In [ ]:
bn.cpds = [cpd_pollution, cpd_smoker, cpd_cancer, cpd_xray, cpd_dyspnoea]

`check_model()` validates that all CPDs are consistent: probabilities are in
[0, 1], each CPD covers every parent configuration, and every node in the
state dictionary appears in exactly one CPD.

In [ ]:
bn.check_model()
print("Model is valid.")

## Inspecting the Network

In [ ]:
print("Edges:", bn.edges)
print()
for cpd in bn.cpds:
    print(f"{cpd.node}  (parents={cpd.parents})")
    for key, val in cpd.values.items():
        print(f"  {key}: {val}")

## Alternative CPD Formats

CPD values can also be specified as lists instead of nested dicts. Lists are
interpreted in the order of the node's state list. This is more concise but
less self-documenting.

In [ ]:
# List-based CPD: equivalent to the dict-based cpd_xray above.
# The inner lists correspond to ["positive", "negative"] in that order.
cpd_xray_list = DiscreteCPD(
    node="Xray",
    parents=["Cancer"],
    values={"yes": [0.9, 0.1], "no": [0.2, 0.8]},
)

# After normalization against the network, the list form is expanded to dicts:
normalized = cpd_xray_list.normalize(bn)
print("Normalized values:", normalized.values)

## A Minimal Example

For a quick two-node network, everything can be done in a few lines.
States and CPDs can be passed directly to the constructor.

In [ ]:
simple_bn = DiscreteBayesianNetwork(
    states={"A": [0, 1], "B": [0, 1]},
    cpds=[
        DiscreteCPD(node="A", values=[0.9, 0.1]),
        DiscreteCPD(node="B", parents=["A"], values={0: [0.2, 0.8], 1: [0.9, 0.1]}),
    ],
)
simple_bn.check_model()

print("Edges: ", simple_bn.edges)
print("States:", simple_bn.states)